# 04 · Resultados, Custo e Explicabilidade

Analisamos o melhor modelo em profundidade:
 análise de custo financeiro, SHAP e preparação para deploy.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib, json, shap

from src.models.evaluate import cost_analysis
from src.visualization.plots import plot_cost_analysis, plot_feature_importance


## 1. Carrega Modelo e Dados


In [ ]:
xgb           = joblib.load('../models/xgb.pkl')
feature_names  = joblib.load('../models/features.pkl')

X_test  = pd.read_csv('../data/processed/X_test.csv')
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()
y_probs = xgb.predict_proba(X_test)[:, 1]

print('Artefatos carregados ✅')


## 2. Análise de Custo Financeiro

Encontramos o threshold que minimiza o custo total,
levando em conta o custo de cada tipo de erro.


In [ ]:
df_thresh = cost_analysis(y_test, y_probs)
best_thresh = df_thresh.loc[df_thresh['custo'].idxmin(), 'threshold']


In [ ]:
plot_cost_analysis(df_thresh, best_thresh)


## 3. Salva Threshold Otimizado


In [ ]:
joblib.dump(best_thresh, '../models/threshold_otimo.pkl')
print(f'Threshold {best_thresh:.2f} salvo em models/threshold_otimo.pkl ✅')


## 4. Explicabilidade com SHAP

**Bar plot**: importância média global.
**Beeswarm**: direção e magnitude do impacto por transação.


In [ ]:
explainer   = shap.Explainer(xgb)
shap_values = explainer(X_test[:200])  # 200 amostras por performance

print('=== Importância Global (médias) ===')
shap.plots.bar(shap_values, max_display=15)


In [ ]:
print('=== Beeswarm: Direção e Magnitude ===')
shap.plots.beeswarm(shap_values, max_display=15)


## 5. Importância das Features (nativa do XGBoost)


In [ ]:
plot_feature_importance(xgb, feature_names, top_n=20)


## 6. Simulação da API

Testa a lógica da API sem precisar subir o servidor.


In [ ]:
threshold_api = joblib.load('../models/threshold_otimo.pkl')

# Pega uma fraude real do conjunto de teste
idx = X_test[y_test == 1].index[0]
transacao = X_test.loc[idx].to_dict()

X_ex  = np.array([transacao[f] for f in feature_names]).reshape(1, -1)
prob  = float(xgb.predict_proba(X_ex)[0][1])
nivel = 'alto' if prob >= 0.7 else ('médio' if prob >= 0.3 else 'baixo')

import json as json_lib
print('=== Resposta da API (simulada) ===')
print(json_lib.dumps({
    'probabilidade_fraude': round(prob, 4),
    'eh_fraude':            bool(prob >= threshold_api),
    'threshold_usado':      round(float(threshold_api), 2),
    'nivel_risco':          nivel,
    'rotulo_real':          int(y_test.loc[idx]),
}, indent=2, ensure_ascii=False))
